# Template Kosongan: Import → EDA → Data Cleaning → Export (Marimo Style)

> **Catatan Penting:** Notebook **Marimo** aslinya berformat `.py` (`marimo edit nama_file.py`). 
> File `.ipynb` ini disusun sebagai **template kosongan terstruktur** yang siap dibuka di Jupyter/VS Code **atau** dikonversi langsung ke format Marimo dengan perintah:
> ```bash
> marimo convert template_eda_cleaning_export_marimo.ipynb -o template_eda_cleaning_export_marimo.py
> ```

## 0. Setup & Import Library

In [ ]:
import marimo as mo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## 1. Import Data (Kosongan — Isi Path Dataset Anda)

In [ ]:
# Isi path file sumber Anda di bawah ini (CSV, Excel, JSON, atau Parquet)
INPUT_PATH = Path("")  # Contoh: Path(r"C:\data\dataset.csv")
CSV_DELIMITER = None    # None = auto-detect, atau isi "," / ";"
CSV_ENCODING = "utf-8-sig"
EXCEL_SHEET = 0

def load_dataset(path: Path) -> pd.DataFrame:
    if not str(path) or not path.is_file():
        print("Status: Path belum diisi atau file tidak ditemukan.")
        return pd.DataFrame()
    ext = path.suffix.lower()
    if ext == ".csv":
        return pd.read_csv(path, sep=CSV_DELIMITER, engine="python", encoding=CSV_ENCODING)
    if ext in {".xlsx", ".xls"}:
        return pd.read_excel(path, sheet_name=EXCEL_SHEET)
    if ext == ".json":
        return pd.read_json(path)
    if ext == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Format file tidak didukung: {ext}")

df_raw = load_dataset(INPUT_PATH)
print(f"Dimensi df_raw: {df_raw.shape}")

## 2. Exploratory Data Analysis (EDA) Awal

In [ ]:
def audit_eda(df: pd.DataFrame) -> None:
    if df.empty:
        print("Dataset masih kosong. Muat data pada tahap 1 terlebih dahulu.")
        return
    print("=== RINGKASAN UMUM ===")
    print(f"Baris: {len(df):,} | Kolom: {df.shape[1]} | Duplikat: {df.duplicated().sum():,} | Total Missing: {df.isna().sum().sum():,}")
    print("\n=== PROFIL KOLOM ===")
    profile = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(dropna=True),
    })
    display(profile)
    print("\n=== STATISTIK DESKRIPTIF ===")
    display(df.describe(include="all").T)

audit_eda(df_raw)

In [ ]:
# Area kosong eksplorasi & visualisasi kustom (isi nama kolom setelah data dimuat)
KOLOM_TARGET = ""   # Contoh: "Revenue"
KOLOM_ANALISIS = "" # Contoh: "ProductRelated_Duration"

# if not df_raw.empty and KOLOM_ANALISIS in df_raw.columns:
#     df_raw[KOLOM_ANALISIS].hist(bins=30)
#     plt.title(f"Distribusi {KOLOM_ANALISIS}")
#     plt.show()

## 3. Data Cleaning Terkendali (Salinan di Memori)

In [ ]:
# Konfigurasi aturan cleaning (default False / aman tanpa mengubah data mentah)
TRIM_WHITESPACE = False
DROP_DUPLICATES = False
MISSING_STRATEGY = "none"  # Pilihan: "none", "drop_rows", "impute_median_mode"
COLUMNS_TO_DROP = []       # Contoh: ["kolom_tidak_terpakai"]

def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    cleaned = df.copy(deep=True)
    if TRIM_WHITESPACE:
        for col in cleaned.select_dtypes(include=["object", "string"]).columns:
            cleaned[col] = cleaned[col].map(lambda x: x.strip() if isinstance(x, str) else x)
    if DROP_DUPLICATES:
        cleaned = cleaned.drop_duplicates().copy()
    if MISSING_STRATEGY == "drop_rows":
        cleaned = cleaned.dropna().copy()
    elif MISSING_STRATEGY == "impute_median_mode":
        for col in cleaned.columns:
            if cleaned[col].isna().any():
                if pd.api.types.is_numeric_dtype(cleaned[col]) and not pd.api.types.is_bool_dtype(cleaned[col]):
                    cleaned[col] = cleaned[col].fillna(cleaned[col].median())
                else:
                    mode_vals = cleaned[col].mode(dropna=True)
                    if not mode_vals.empty:
                        cleaned[col] = cleaned[col].fillna(mode_vals.iloc[0])
    valid_drop = [c for c in COLUMNS_TO_DROP if c in cleaned.columns]
    if valid_drop:
        cleaned = cleaned.drop(columns=valid_drop)
    return cleaned

df_clean = clean_data(df_raw)
print(f"Dimensi sebelum cleaning: {df_raw.shape} -> sesudah: {df_clean.shape}")

## 4. Validasi Pasca-Cleaning

In [ ]:
# Cek ulang apakah masih ada missing atau duplikat setelah aturan cleaning diterapkan
if not df_clean.empty:
    print("Duplikat tersisa :", int(df_clean.duplicated().sum()))
    print("Missing tersisa  :", int(df_clean.isna().sum().sum()))
    display(df_clean.head(5))
else:
    print("df_clean masih kosong.")

## 5. Export Data Clean (Kosongan — Isi Path & Format Output)

In [ ]:
# Konfigurasi export hasil bersih (tidak menimpa file sumber)
EXPORT_ENABLED = False
OUTPUT_PATH = Path(r"C:\01.Training\dataclean")  # Bisa folder atau nama file lengkap
EXPORT_FORMAT = "csv"       # Pilihan: "csv", "xlsx", "json", "parquet"

def export_clean_data(df: pd.DataFrame, out_path: Path, fmt: str, enabled: bool = False) -> None:
    if not enabled:
        print("Export belum diaktifkan (EXPORT_ENABLED = False).")
        return
    if df.empty:
        print("Gagal export: df_clean masih kosong.")
        return
    out_path = Path(out_path)
    if str(out_path) in {"", "."}:
        print("Gagal export: OUTPUT_PATH belum diisi.")
        return
    fmt_clean = fmt.strip().lower()
    extensions = {"csv": ".csv", "xlsx": ".xlsx", "json": ".json", "parquet": ".parquet"}
    if fmt_clean not in extensions:
        raise ValueError(f"Format export tidak didukung: {fmt}")
    # Jika OUTPUT_PATH berupa folder (atau belum memiliki ekstensi), buat nama file otomatis.
    if out_path.is_dir() or not out_path.suffix:
        out_path = out_path / f"data_cleaned{extensions[fmt_clean]}"
    if str(INPUT_PATH) and INPUT_PATH.exists() and INPUT_PATH.resolve() == out_path.resolve():
        raise ValueError("OUTPUT_PATH tidak boleh sama dengan INPUT_PATH.")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    if fmt_clean == "csv":
        df.to_csv(out_path, index=False, encoding="utf-8-sig")
    elif fmt_clean == "xlsx":
        df.to_excel(out_path, index=False)
    elif fmt_clean == "json":
        df.to_json(out_path, orient="records", indent=2, force_ascii=False)
    elif fmt_clean == "parquet":
        df.to_parquet(out_path, index=False)
    print(f"Export berhasil: {out_path} ({len(df):,} baris x {df.shape[1]} kolom, format={fmt_clean})")

export_clean_data(df_clean, OUTPUT_PATH, EXPORT_FORMAT, enabled=EXPORT_ENABLED)